# Partie II — CNN et vision par ordinateur
## Classification d'images avec réseaux convolutionnels

**Dataset retenu : *EMNIST Letters*** — 26 classes de lettres manuscrites (28×28, niveaux de gris),
extension de MNIST plus difficile. Ce choix se distingue des datasets proposés (MNIST, Fashion-MNIST,
CIFAR-10) tout en restant un jeu d'images réel, léger et téléchargeable automatiquement.

> **Google Colab** : activer le GPU (`Exécution → Modifier → Paramètres du notebook → GPU`).
> Téléchargement automatique via `torchvision` (avec miroirs de secours).

### Plan
1. Pourquoi un MLP est mal adapté aux images ; idées fondatrices des CNN
2. Calculs manuels (corrélation croisée, tailles de sortie conv et pooling)
3. Implémentation *from scratch* : corrélation 2D, max-pooling, average-pooling + comparaison PyTorch
4. Données EMNIST (préparation, orientation, normalisation)
5. CNN type **LeNet** paramétré
6. Étude expérimentale (padding, stride, pooling, nb de filtres, conv 1×1)
7. Visualisation de cartes de caractéristiques
8. Comparaison MLP vs CNN
9. Analyse critique et question de synthèse


## 1. Pourquoi un MLP est mal adapté aux images

Un MLP traite l'image **aplatie** : pour 28×28 = 784 pixels et une première couche de 256 neurones,
il faut déjà 200 704 poids. Surtout, l'aplatissement **détruit la structure spatiale** (deux pixels
voisins ne sont pas traités différemment de deux pixels éloignés) et le modèle n'est **pas invariant
par translation**.

Les CNN reposent sur trois idées fondatrices :
- **Localité** : un neurone ne regarde qu'un voisinage (champ récepteur), pas toute l'image.
- **Partage des poids** : le même filtre balaie toute l'image → bien moins de paramètres et invariance par translation.
- **Hiérarchie des représentations** : les premières couches détectent des bords/textures, les suivantes
  des motifs de plus en plus abstraits.


In [ ]:
import os, copy, time, random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def try_gpu(i=0):
    if torch.cuda.is_available() and torch.cuda.device_count() >= i + 1:
        return torch.device(f"cuda:{i}")
    return torch.device("cpu")

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
print("PyTorch :", torch.__version__, "| Colab :", IN_COLAB)


In [ ]:
# --- CONFIG centralisée (non hardcodé) ---
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    seed: int = 42
    device: torch.device = field(default_factory=try_gpu)
    data_root: str = "/content/data" if ("google.colab" in str(globals().get("get_ipython", lambda: "")())) else "data"
    save_dir: str = "/content/p2_artifacts" if ("google.colab" in str(globals().get("get_ipython", lambda: "")())) else "p2_artifacts"
    # EMNIST
    emnist_split: str = "letters"
    num_classes: int = 26
    # Plafonds (réduisent le temps de calcul ; à augmenter si besoin)
    max_train_samples: int = 24000
    max_test_samples: int = 4000
    batch_size: int = 128
    # Architecture LeNet (paramétrable)
    c1: int = 6            # filtres conv1
    c2: int = 16           # filtres conv2
    padding: int = 2       # padding conv1
    stride: int = 1
    pool: str = "max"      # "max" | "avg"
    use_1x1: bool = False  # ajoute une conv 1x1 après conv2
    fc_hidden: List[int] = field(default_factory=lambda: [120, 84])
    # Entraînement
    lr: float = 1e-3
    epochs: int = 4
    epochs_experiment: int = 2   # pour l'étude comparative (plus court)

CFG = Config()
os.makedirs(CFG.save_dir, exist_ok=True)
set_seed(CFG.seed)
print("Device :", CFG.device)


## 2. Calculs manuels

Pour une entrée de taille $n \times n$, un noyau $k \times k$, un *padding* $p$ et un *stride* $s$, la
**taille de sortie** d'une convolution (corrélation croisée) est :

$$ n_{out} = \left\lfloor \frac{n - k + 2p}{s} \right\rfloor + 1 $$

La même formule s'applique au **pooling** (avec $p=0$ en général).

**Exemples (vérifiés numériquement plus bas) :**
- Entrée 28×28, noyau 5×5, $p=2$, $s=1$ → $(28-5+4)/1 + 1 = 28$ (taille conservée).
- Entrée 28×28, noyau 5×5, $p=0$, $s=1$ → $24$.
- Pooling 2×2, $s=2$ sur 28×28 → $(28-2)/2 + 1 = 14$ (réduction de moitié).

**Corrélation croisée 2D** (sortie en position $(i,j)$) :
$$ Y_{i,j} = \sum_{a=0}^{k-1}\sum_{b=0}^{k-1} X_{i+a,\,j+b}\, K_{a,b} $$


In [ ]:
def conv_out_size(n, k, p=0, s=1):
    return (n - k + 2 * p) // s + 1

for (n, k, p, s) in [(28, 5, 2, 1), (28, 5, 0, 1), (28, 2, 0, 2)]:
    print(f"n={n}, k={k}, p={p}, s={s}  ->  sortie = {conv_out_size(n, k, p, s)}")


## 3. Implémentation *from scratch* et comparaison avec PyTorch

On programme la corrélation croisée 2D, le max-pooling et l'average-pooling « à la main », puis on
vérifie l'**égalité numérique** avec les opérations PyTorch correspondantes.


In [ ]:
def corr2d(X, K):
    "Corrélation croisée 2D (stride 1, sans padding)."
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

def pool2d(X, pool_size, mode="max"):
    "Pooling 2D (stride 1) max ou moyenne, implémenté à la main."
    ph, pw = pool_size
    Y = torch.zeros((X.shape[0] - ph + 1, X.shape[1] - pw + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            window = X[i:i + ph, j:j + pw]
            Y[i, j] = window.max() if mode == "max" else window.mean()
    return Y

X = torch.arange(25, dtype=torch.float32).reshape(5, 5)
K = torch.tensor([[1., 0.], [0., -1.]])

# Référence PyTorch (conv2d réalise une corrélation croisée)
Y_manual = corr2d(X, K)
Y_torch = F.conv2d(X.view(1, 1, 5, 5), K.view(1, 1, 2, 2)).view(4, 4)
print("Corrélation croisée — égalité PyTorch :", torch.allclose(Y_manual, Y_torch))

Ymax_manual = pool2d(X, (2, 2), "max")
Ymax_torch = F.max_pool2d(X.view(1, 1, 5, 5), 2, stride=1).view(4, 4)
print("Max-pooling — égalité PyTorch        :", torch.allclose(Ymax_manual, Ymax_torch))

Yavg_manual = pool2d(X, (2, 2), "avg")
Yavg_torch = F.avg_pool2d(X.view(1, 1, 5, 5), 2, stride=1).view(4, 4)
print("Avg-pooling — égalité PyTorch        :", torch.allclose(Yavg_manual, Yavg_torch))


## 4. Préparation des données EMNIST Letters

Les images EMNIST sont stockées **tournées et miroitées** : on applique une transposition pour les
remettre droites, puis `ToTensor` et une normalisation. Les labels `letters` vont de 1 à 26 → on
soustrait 1. On sous-échantillonne (plafonds de `CFG`) pour accélérer les expériences.


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Correction d'orientation propre à EMNIST + normalisation
emnist_tf = transforms.Compose([
    lambda img: transforms.functional.hflip(transforms.functional.rotate(img, -90)),
    transforms.ToTensor(),
    transforms.Normalize((0.1736,), (0.3317,)),
])
target_tf = transforms.Lambda(lambda y: y - 1)   # 1..26 -> 0..25

# Miroirs de secours pour le téléchargement EMNIST
EMNIST_MIRRORS = [
    "https://biometrics.nist.gov/cs_links/EMNIST/gzip.zip",
    "https://www.itl.nist.gov/iaui/vip/cs_links/EMNIST/gzip.zip",
    "https://cloudstor.aarnet.edu.au/plus/serve/static/EMNIST/gzip.zip",
]

def load_emnist(train):
    last_err = None
    for url in EMNIST_MIRRORS:
        try:
            datasets.EMNIST.url = url
            return datasets.EMNIST(root=CFG.data_root, split=CFG.emnist_split, train=train,
                                   download=True, transform=emnist_tf, target_transform=target_tf)
        except Exception as e:
            last_err = e
            print("Échec miroir", url, "->", str(e)[:80])
    raise RuntimeError(f"Téléchargement EMNIST impossible : {last_err}")

full_train = load_emnist(True)
full_test = load_emnist(False)

def subsample(ds, n, seed=CFG.seed):
    if n is None or n >= len(ds):
        return ds
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(ds), generator=g)[:n].tolist()
    return Subset(ds, idx)

train_ds = subsample(full_train, CFG.max_train_samples)
test_ds = subsample(full_test, CFG.max_test_samples)
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False)
print(f"Train={len(train_ds)}  Test={len(test_ds)}  Classes={CFG.num_classes}")


In [ ]:
# Aperçu de quelques images (lettres a..z)
xb, yb = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for ax, img, lab in zip(axes.flat, xb, yb):
    ax.imshow(img.squeeze(), cmap="gray"); ax.set_title(chr(ord('a') + int(lab))); ax.axis("off")
plt.suptitle("EMNIST Letters — échantillon"); plt.tight_layout(); plt.show()


## 5. CNN type LeNet (paramétrable)

L'architecture est pilotée par `CFG` : nombre de filtres (`c1`, `c2`), `padding`, `stride`, type de
`pool`, ajout optionnel d'une **convolution 1×1** et tailles des couches denses. La dimension d'entrée du
classifieur est déduite automatiquement (`LazyLinear`), ce qui rend le modèle robuste aux changements de
géométrie.


In [ ]:
class LeNet(nn.Module):
    def __init__(self, cfg, num_classes):
        super().__init__()
        Pool = nn.MaxPool2d if cfg.pool == "max" else nn.AvgPool2d
        layers = [
            nn.Conv2d(1, cfg.c1, kernel_size=5, padding=cfg.padding, stride=cfg.stride),
            nn.ReLU(), Pool(2),
            nn.Conv2d(cfg.c1, cfg.c2, kernel_size=5),
            nn.ReLU(),
        ]
        if cfg.use_1x1:                       # convolution 1x1 : mélange de canaux, peu de paramètres
            layers += [nn.Conv2d(cfg.c2, cfg.c2, kernel_size=1), nn.ReLU()]
        layers += [Pool(2), nn.Flatten()]
        self.features = nn.Sequential(*layers)
        dense = []
        for h in cfg.fc_hidden:
            dense += [nn.LazyLinear(h), nn.ReLU()]
        dense += [nn.LazyLinear(num_classes)]
        self.classifier = nn.Sequential(*dense)
    def forward(self, x):
        return self.classifier(self.features(x))

# Initialisation des couches Lazy via un passage de données
model = LeNet(CFG, CFG.num_classes).to(CFG.device)
_ = model(xb.to(CFG.device))
n_params = sum(p.numel() for p in model.parameters())
print(model); print("Paramètres :", n_params)


In [ ]:
def evaluate(model, loader, device):
    model.eval(); correct = n = 0
    crit = nn.CrossEntropyLoss(); tot_loss = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            tot_loss += crit(out, yb).item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item(); n += len(yb)
    return tot_loss / n, correct / n

def train_cnn(model, train_loader, test_loader, cfg, device, epochs=None, verbose=True):
    epochs = epochs or cfg.epochs
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    crit = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "test_acc": []}
    t0 = time.time()
    for ep in range(epochs):
        model.train(); run = 0.0; n = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
            run += loss.item() * len(yb); n += len(yb)
        tl, ta = evaluate(model, test_loader, device)
        hist["train_loss"].append(run / n); hist["test_acc"].append(ta)
        if verbose:
            print(f"  ep {ep+1}/{epochs} | train {run/n:.4f} | test_acc {ta:.4f}")
    hist["time"] = time.time() - t0
    return hist

set_seed(CFG.seed)
model = LeNet(CFG, CFG.num_classes).to(CFG.device); _ = model(xb.to(CFG.device))
hist = train_cnn(model, train_loader, test_loader, CFG, CFG.device)
torch.save(model.state_dict(), os.path.join(CFG.save_dir, "lenet_best.params"))
print("Accuracy test finale :", round(hist["test_acc"][-1], 4))


## 6. Étude expérimentale des choix architecturaux

On fait varier un facteur à la fois (padding, stride, type de pooling, nombre de filtres, présence d'une
conv 1×1) et on compare **accuracy de test, nombre de paramètres et temps d'entraînement** (entraînement
volontairement court, `epochs_experiment`).


In [ ]:
from dataclasses import replace
import pandas as pd

variants = {
    "baseline":         dict(),
    "padding=0":        dict(padding=0),
    "stride=2":         dict(stride=2),
    "avg_pool":         dict(pool="avg"),
    "filtres x2":       dict(c1=12, c2=32),
    "conv_1x1":         dict(use_1x1=True),
}

rows = []
for name, override in variants.items():
    cfg_v = replace(CFG, **override)
    set_seed(CFG.seed)
    m = LeNet(cfg_v, CFG.num_classes).to(CFG.device); _ = m(xb.to(CFG.device))
    h = train_cnn(m, train_loader, test_loader, cfg_v, CFG.device,
                  epochs=CFG.epochs_experiment, verbose=False)
    rows.append({"variante": name,
                 "test_acc": round(h["test_acc"][-1], 4),
                 "params": sum(p.numel() for p in m.parameters()),
                 "temps_s": round(h["time"], 1)})
    print(f"{name:12s} -> acc={h['test_acc'][-1]:.4f}")

df = pd.DataFrame(rows).set_index("variante")
df


**Lecture des résultats.** Le *padding* préserve la taille spatiale et donc l'information de bord ;
un *stride* élevé réduit fortement la résolution (moins de paramètres mais perte de détail, accuracy en
baisse) ; le *max-pooling* est généralement plus discriminant que l'*average-pooling* sur des traits fins
de lettres ; augmenter le nombre de filtres accroît la capacité (et le coût) ; la **conv 1×1** recombine
les canaux à très faible coût paramétrique. Le tableau ci-dessus permet d'arbitrer **performance vs coût**.


## 7. Visualisation des cartes de caractéristiques

On affiche les activations de la première couche convolutive sur un exemple : chaque carte met en évidence
un type de motif (bords, orientations) détecté par un filtre.


In [ ]:
sample = xb[0:1].to(CFG.device)
first_conv = model.features[0]   # première Conv2d
with torch.no_grad():
    fmaps = torch.relu(first_conv(sample)).cpu().squeeze(0)

n = fmaps.shape[0]
fig, axes = plt.subplots(1, n + 1, figsize=(2 * (n + 1), 2.2))
axes[0].imshow(xb[0].squeeze(), cmap="gray"); axes[0].set_title("entrée"); axes[0].axis("off")
for k in range(n):
    axes[k + 1].imshow(fmaps[k], cmap="viridis"); axes[k + 1].set_title(f"filtre {k}"); axes[k + 1].axis("off")
plt.suptitle("Cartes de caractéristiques — 1ère couche conv"); plt.tight_layout(); plt.show()


## 8. Comparaison MLP vs CNN sur le même dataset

On entraîne un MLP simple (images aplaties) et on le compare au CNN, à budget d'époques comparable.


In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, num_classes, hidden=256):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.LazyLinear(hidden), nn.ReLU(),
                                 nn.LazyLinear(hidden), nn.ReLU(), nn.LazyLinear(num_classes))
    def forward(self, x): return self.net(x)

set_seed(CFG.seed)
mlp = SimpleMLP(CFG.num_classes).to(CFG.device); _ = mlp(xb.to(CFG.device))
h_mlp = train_cnn(mlp, train_loader, test_loader, CFG, CFG.device, epochs=CFG.epochs, verbose=False)

print(f"MLP  : acc={h_mlp['test_acc'][-1]:.4f} | params={sum(p.numel() for p in mlp.parameters())}")
print(f"CNN  : acc={hist['test_acc'][-1]:.4f} | params={n_params}")

plt.plot(hist["test_acc"], "-o", label="CNN")
plt.plot(h_mlp["test_acc"], "-s", label="MLP")
plt.xlabel("époque"); plt.ylabel("accuracy test"); plt.legend(); plt.title("CNN vs MLP — EMNIST"); plt.show()


## 9. Analyse critique

- **Supériorité du CNN.** À nombre de paramètres souvent **inférieur**, le CNN dépasse le MLP : le partage
  de poids et la localité exploitent la structure 2D des lettres, là où le MLP doit tout réapprendre sans a priori spatial.
- **Effets architecturaux.** Les expériences confirment les calculs dimensionnels : le *stride* et l'absence
  de *padding* réduisent la résolution (donc l'information disponible), tandis que davantage de filtres ou
  une conv 1×1 augmentent la capacité de représentation à des coûts variables.
- **Limites.** EMNIST Letters comporte des confusions intrinsèques (ex. `i`/`l`, `o`/lettres rondes) visibles
  dans les erreurs ; l'entraînement court et le sous-échantillonnage bornent l'accuracy atteignable.

## Question de synthèse — Partie II

> *Pourquoi un CNN est-il plus pertinent qu'un MLP pour la classification d'images, et comment padding,
> stride, pooling et profondeur influencent-ils les performances ?*

Le CNN encode des **a priori adaptés aux images** : localité (champs récepteurs), partage de poids
(invariance par translation, économie de paramètres) et hiérarchie de représentations. Le MLP, en aplatissant
l'image, perd la géométrie et explose en nombre de paramètres. Les hyperparamètres convolutionnels agissent
directement sur la **taille des cartes** via $n_{out}=\lfloor (n-k+2p)/s \rfloor +1$ : le *padding* préserve
les bords, le *stride* sous-échantillonne agressivement, le *pooling* apporte robustesse et invariance locale
(max vs moyenne selon que l'on privilégie les traits saillants ou la régularité), et la profondeur permet de
composer des motifs simples en concepts complexes — au prix d'un coût de calcul et d'un risque de
sur-apprentissage qu'il faut contrôler.
